In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization, Multiply
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, accuracy_score
import tensorflow.keras.backend as K

In [ ]:
def focal_loss(gamma=2., alpha=0.5):
    def loss(y_true, y_pred):
        y_pred = K.clip(y_pred, 1e-7, 1 - 1e-7)
        pt = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        return -K.mean(alpha * K.pow(1 - pt, gamma) * K.log(pt))
    return loss

In [ ]:
def preprocess_image(img):
    img = img / 255.0
    img = tf.image.adjust_contrast(img, 1.5)
    return img

In [ ]:
data_dir = "/kaggle/input/datasets/afridirahman/brain-stroke-ct-image-dataset/Brain_Data_Organised"

In [ ]:
img_size = (224, 224)
batch_size = 16

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_image,
    validation_split=0.2,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

train_data = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

val_data = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

In [ ]:
class_weights = {0: 1.0, 1: 1.4}

In [ ]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))

# Freeze layers
for layer in base_model.layers[:-50]:
    layer.trainable = False

for layer in base_model.layers[-50:]:
    layer.trainable = True

# Feature extraction
x = base_model.output
x = GlobalAveragePooling2D()(x)

# 🔥 ATTENTION (FIXED SHAPE)
feature_dim = x.shape[-1]   # should be 2048

attention = Dense(feature_dim // 2, activation='relu')(x)
attention = Dense(feature_dim, activation='sigmoid')(attention)

x = Multiply()([x, attention])

# Dense layers
x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)

output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.00005),
    loss=focal_loss(),
    metrics=['accuracy']
)

model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

lr_reduce = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3,
    min_lr=1e-6
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=25,
    class_weight=class_weights,
    callbacks=[early_stop, lr_reduce]
)

In [ ]:
y_pred_prob = model.predict(val_data)
y_pred = (y_pred_prob > 0.6).astype(int)

print("Accuracy:", accuracy_score(val_data.classes, y_pred))
print(classification_report(val_data.classes, y_pred))